# The galaxy collision

The project's demonstration scenario, run from Python. Two disc galaxies of
unequal mass on a bound, grazing encounter: they fall together, pass, draw tidal
tails out of one another, separate, return and merge.

This notebook runs a small version of it so that it finishes in a couple of
minutes on a laptop. The real thing is twenty thousand particles watched live in
the viewer, which is what `docs/visualisation.md` describes; what is here is the
same scenario with the state as NumPy arrays instead of pixels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import orrery

configuration = orrery.Configuration()
configuration.initial_conditions.kind = orrery.InitialConditionKind.galaxy_collision
configuration.initial_conditions.count = 4000
configuration.initial_conditions.total_mass = 1.0
configuration.initial_conditions.mass_ratio = 0.5
configuration.initial_conditions.separation = 20.0
configuration.initial_conditions.impact_parameter = 2.0
configuration.initial_conditions.approach_speed = 0.8
configuration.initial_conditions.secondary_inclination = 1.0

configuration.solver.kind = orrery.SolverKind.barnes_hut
configuration.solver.softening = 0.05
configuration.solver.opening_angle = 0.6

configuration.integrator.kind = orrery.IntegratorKind.velocity_verlet
configuration.run.timestep = 1.0 / 128.0
configuration.run.steps = 6000
configuration.run.seed = 20260812

assert orrery.problems_with(configuration) == []

# Which particles came from which galaxy. Without this a merged remnant is one
# undifferentiated cloud, and the question a collision simulation is asked is
# where the material ended up.
primary = orrery.primary_galaxy_count(configuration)
print(f"{configuration.initial_conditions.count} particles, {primary} in the larger galaxy")

## The run

Six thousand steps, sampling the state every few hundred. The positions are read
through NumPy views and copied only into the frames that are kept, which is the
one place a copy is wanted: a frame has to outlive the step that produced it.

In [ ]:
simulation = orrery.assemble(configuration)
print(f"{simulation.solver_name} solver, {simulation.integrator_name} integrator")

sample_every = 500
frames, times, diagnostics = [], [], []


def record():
    frames.append(orrery.stacked(simulation.particles))
    times.append(simulation.time)
    diagnostics.append(simulation.measure())


record()
while simulation.step_index < configuration.run.steps:
    simulation.run(min(sample_every, configuration.run.steps - simulation.step_index))
    record()

print(f"{len(frames)} frames over {simulation.time:.1f} time units")

In [ ]:
chosen = np.linspace(0, len(frames) - 1, 6).astype(int)

figure, axes = plt.subplots(2, 3, figsize=(13, 8.5))
for panel, index in zip(axes.flat, chosen):
    positions = frames[index]
    panel.scatter(
        positions[:primary, 0], positions[:primary, 1], s=0.7, alpha=0.5, c="#ffd27f",
        linewidths=0,
    )
    panel.scatter(
        positions[primary:, 0], positions[primary:, 1], s=0.7, alpha=0.5, c="#7fb8ff",
        linewidths=0,
    )
    panel.set_facecolor("black")
    panel.set_xlim(-18, 18)
    panel.set_ylim(-18, 18)
    panel.set_aspect("equal")
    panel.set_xticks([])
    panel.set_yticks([])
    panel.set_title(f"t = {times[index]:.1f}", fontsize=10)

figure.suptitle("two disc galaxies on a bound, grazing encounter")
plt.tight_layout()
plt.show()

## What the run conserved, and what it settled at

Two numbers say whether the run is worth looking at. The energy error says
whether the timestep and the softening were adequate; the virial ratio says what
the remnant is doing, since a merger that has virialised has settled at one.

In [ ]:
energies = np.array([d.total_energy for d in diagnostics])
virial = np.array([d.virial_ratio for d in diagnostics])
drift = np.abs((energies - energies[0]) / energies[0])

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(times, drift)
axes[0].set_xlabel("time")
axes[0].set_ylabel("relative energy error")
axes[0].set_title(f"energy conserved to {drift.max():.2e}")
axes[0].grid(alpha=0.3)

axes[1].plot(times, virial)
axes[1].axhline(1.0, color="grey", linestyle="--", linewidth=1)
axes[1].set_xlabel("time")
axes[1].set_ylabel("2T / |U|")
axes[1].set_title("the merger virialising")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"energy error at the end: {drift[-1]:.3e}")
print(f"virial ratio: {virial[0]:.3f} at the start, {virial[-1]:.3f} at the end")

# A run whose energy has moved by a per cent is not a run to draw conclusions
# from, whatever the picture looks like.
assert drift.max() < 1e-2

## Where the material ended up

The question a collision is asked. The two galaxies were tagged at the start, so
the answer is a matter of counting rather than of guessing from a picture.

In [ ]:
final = frames[-1]

# The two galaxies carry different particle masses, so the centre of the remnant
# is a mass-weighted mean rather than an average of positions.
mass = np.asarray(simulation.particles.mass)
centre = np.average(final, axis=0, weights=mass)
distance = np.linalg.norm(final - centre, axis=1)

bound = distance < 5.0
print(f"{bound.sum()} of {len(final)} particles are within 5 length units of the remnant")
print(f"  from the larger galaxy: {bound[:primary].sum()} of {primary}")
print(f"  from the smaller one:   {bound[primary:].sum()} of {len(final) - primary}")

figure, axes = plt.subplots(figsize=(7, 4.5))
edges = np.logspace(-1, 1.6, 40)
axes.hist(distance[:primary], bins=edges, histtype="step", label="larger galaxy", linewidth=1.5)
axes.hist(distance[primary:], bins=edges, histtype="step", label="smaller galaxy", linewidth=1.5)
axes.set_xscale("log")
axes.set_xlabel("distance from the remnant's centre")
axes.set_ylabel("particles")
axes.legend()
axes.grid(alpha=0.3)
plt.show()

## The same run from the command line

Nothing here is reachable only from Python. The configuration this notebook
built can be written out and run by the command-line program, or watched live in
the viewer, and it is the same run either way.

In [ ]:
document = orrery.write_configuration(configuration)
print(document)

# And it reads back as itself, which is what makes a run reproducible from a
# document plus a revision of this repository.
assert orrery.parse_configuration(document, "notebook") == configuration

Save that to `collision.orrery` and:

```
orrery run collision.orrery
orrery-view play collision.otj
```

`examples/collision.orrery` is the full-sized version of it.